# Video Captioning Pipeline Runner

This notebook is a thin orchestration layer over the repo scripts. It does **not** reimplement the pipeline. Each stage calls the existing `.py` entrypoints with explicit arguments so you can run the full workflow from raw metadata and videos through training and category analysis.

Recommended usage:
1. Set the configuration in the next cell.
2. Run the helper cell.
3. Run the command-preview cell.
4. Run the stages you need, or run the full pipeline cell.

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / 'train_final_bart.py').exists(), 'Open this notebook from the repo root.'
PYTHON = sys.executable

DATASET_MODE = 'subset'   # 'subset' or 'full'
DEVICE = 'mps'            # 'mps', 'cpu', 'cuda', or 'auto'
RUN_ALL_STEPS = False     # Set True only when you really want the full pipeline to execute.

# Stage toggles for the "run all" cell.
RUN_CAPTION_PREP = True
RUN_TASCC = False
RUN_VISUAL_SPLIT = True
RUN_AUDIO_WAV = False
RUN_AUDIO_VGGISH = False
RUN_MULTIMODAL_CACHE = False
RUN_TRAIN = True
RUN_CATEGORY_ANALYSIS = True
RUN_ABLATIONS = False

# Training config.
TRAIN_EPOCHS = 40
TRAIN_BATCH_SIZE = 2
TRAIN_GRAD_ACCUM = 4
EVAL_BATCH_SIZE = 1
VAL_TEST_BEAMS = 2

# Output naming.
RUN_NAME = f'notebook_final_{DATASET_MODE}'
TRAIN_RUN_DIR = REPO_ROOT / 'outputs' / RUN_NAME
CATEGORY_OUTPUT_DIR = TRAIN_RUN_DIR / 'category_breakdown_xe'

DATASET_ROOT = REPO_ROOT / 'dataset' / 'MSR-VTT'


def first_existing(*candidates: Path) -> Path:
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


SUBSET_VIDEO_DIR = first_existing(
    DATASET_ROOT / 'downsampled_2500_videos',
)
FULL_TRAINVAL_VIDEO_DIR = first_existing(
    DATASET_ROOT / 'TrainValVideo',
    DATASET_ROOT / 'full_dataset' / 'TrainValVideo',
    DATASET_ROOT / 'full_dataset',
)
FULL_TEST_VIDEO_DIR = first_existing(
    DATASET_ROOT / 'TestVideo',
    DATASET_ROOT / 'full_dataset' / 'TestVideo',
)

SUBSET_FUSED_DIR = REPO_ROOT / 'features' / 'tascc_fused'
SUBSET_TS_DIR = REPO_ROOT / 'features' / 'tascc_timestamps'
FULL_FUSED_DIR = REPO_ROOT / 'datas' / 'feats' / 'tascc_fused'
FULL_TS_DIR = REPO_ROOT / 'datas' / 'feats' / 'tascc_timestamps' 


In [ ]:
def run_cmd(cmd, cwd=REPO_ROOT):
    pretty = ' '.join(shlex.quote(str(part)) for part in cmd)
    print(f'$ {pretty}')
    subprocess.run([str(part) for part in cmd], cwd=cwd, check=True)


def tascc_commands(mode: str):
    if mode == 'subset':
        return [[
            PYTHON,
            REPO_ROOT / 'prepro_tascc_feats.py',
            '--video-dir', SUBSET_VIDEO_DIR,
            '--video-list-json', DATASET_ROOT / 'downsampled_2500.json',
            '--output-dir', SUBSET_FUSED_DIR,
            '--timestamps-dir', SUBSET_TS_DIR,
            '--manifest-path', REPO_ROOT / 'features' / 'tascc_manifest_subset.json',
        ]]
    return [
        [
            PYTHON,
            REPO_ROOT / 'prepro_tascc_feats.py',
            '--video-dir', FULL_TRAINVAL_VIDEO_DIR,
            '--video-list-json', DATASET_ROOT / 'train_val_videodatainfo.json',
            '--output-dir', FULL_FUSED_DIR,
            '--timestamps-dir', FULL_TS_DIR,
            '--manifest-path', REPO_ROOT / 'datas' / 'feats' / 'tascc_manifest_trainval_full.json',
        ],
        [
            PYTHON,
            REPO_ROOT / 'prepro_tascc_feats.py',
            '--video-dir', FULL_TEST_VIDEO_DIR,
            '--video-list-json', DATASET_ROOT / 'test_videodatainfo.json',
            '--output-dir', FULL_FUSED_DIR,
            '--timestamps-dir', FULL_TS_DIR,
            '--manifest-path', REPO_ROOT / 'datas' / 'feats' / 'tascc_manifest_test_full.json',
        ],
    ]


def split_visual_command(mode: str):
    cmd = [PYTHON, REPO_ROOT / 'data' / 'split_visual_embeddings.py', '--dataset-mode', mode]
    if mode == 'full':
        cmd.extend(['--fused-dir', FULL_FUSED_DIR])
    return cmd


def train_command(mode: str):
    return [
        PYTHON,
        REPO_ROOT / 'train_final_bart.py',
        '--architecture', 'stable',
        '--dataset-mode', mode,
        '--modalities', 'clip_dino_audio',
        '--visual-source', 'auto',
        '--decoder-train-mode', 'freeze',
        '--xe-epochs', str(TRAIN_EPOCHS),
        '--skip-scst',
        '--batch-size', str(TRAIN_BATCH_SIZE),
        '--grad-accum-steps', str(TRAIN_GRAD_ACCUM),
        '--eval-batch-size', str(EVAL_BATCH_SIZE),
        '--encoder-lr', '1e-4',
        '--bart-lr', '2e-5',
        '--val-num-beams', str(VAL_TEST_BEAMS),
        '--test-num-beams', str(VAL_TEST_BEAMS),
        '--device', DEVICE,
        '--run-dir', TRAIN_RUN_DIR,
    ]


def category_analysis_command(mode: str):
    return [
        PYTHON,
        REPO_ROOT / 'analyze_test_by_category.py',
        '--predictions', TRAIN_RUN_DIR / 'test_predictions_xe.json',
        '--train-dataset-mode', mode,
        '--output-dir', CATEGORY_OUTPUT_DIR,
    ]


def ablation_command():
    return [
        PYTHON,
        REPO_ROOT / 'ablation' / 'run_ablation_suite.py',
        '--resource-root', REPO_ROOT,
        '--output-root', REPO_ROOT / 'outputs' / 'ablation_suite_notebook',
        '--device', DEVICE,
        '--epochs-full', '20',
        '--epochs-subset', '20',
        '--batch-size-full', '8',
        '--batch-size-subset', '16',
        '--skip-existing',
    ]


## Command Preview

This cell prints the exact commands that the notebook will run for the current configuration.

In [ ]:
print('Caption prep:')
print(' ', ' '.join(map(str, [PYTHON, REPO_ROOT / 'data' / 'extract_captions.py', '--dataset-mode', DATASET_MODE])))
print(' ', ' '.join(map(str, [PYTHON, REPO_ROOT / 'data' / 'build_vocab.py', '--dataset-mode', DATASET_MODE])))

print('\nTASCC:')
for cmd in tascc_commands(DATASET_MODE):
    print(' ', ' '.join(map(str, cmd)))

print('\nSplit visual embeddings:')
print(' ', ' '.join(map(str, split_visual_command(DATASET_MODE))))

print('\nAudio WAV:')
print(' ', ' '.join(map(str, [PYTHON, REPO_ROOT / 'data' / 'extract_audio_wav.py', '--dataset-mode', DATASET_MODE])))

print('\nAudio VGGish:')
print(' ', ' '.join(map(str, [PYTHON, REPO_ROOT / 'data' / 'extract_vggish_embeddings.py', '--dataset-mode', DATASET_MODE])))

print('\nMultimodal cache:')
print(' ', ' '.join(map(str, [PYTHON, REPO_ROOT / 'data' / 'extract_multimodal_embeddings.py', '--dataset-mode', DATASET_MODE])))

print('\nTraining:')
print(' ', ' '.join(map(str, train_command(DATASET_MODE))))

print('\nCategory analysis:')
print(' ', ' '.join(map(str, category_analysis_command(DATASET_MODE))))

print('\nAblations:')
print(' ', ' '.join(map(str, ablation_command())))


## Stage 1: Build captions and vocabulary

In [ ]:
run_cmd([PYTHON, REPO_ROOT / 'data' / 'extract_captions.py', '--dataset-mode', DATASET_MODE])
run_cmd([PYTHON, REPO_ROOT / 'data' / 'build_vocab.py', '--dataset-mode', DATASET_MODE])

## Stage 2: Run TASCC feature extraction

Set `RUN_TASCC = True` only if you need to build fused TASCC features from videos. If the fused features already exist, skip this stage.

In [ ]:
if RUN_TASCC:
    for cmd in tascc_commands(DATASET_MODE):
        run_cmd(cmd)
else:
    print('Skipping TASCC extraction because RUN_TASCC is False.')

## Stage 3: Build split CLIP/DINO feature stores

In [ ]:
if RUN_VISUAL_SPLIT:
    run_cmd(split_visual_command(DATASET_MODE))
else:
    print('Skipping split visual embedding build.')

## Stage 4: Build audio artifacts

`extract_audio_wav.py` requires `ffmpeg` and `ffprobe` to be installed.

In [ ]:
if RUN_AUDIO_WAV:
    run_cmd([PYTHON, REPO_ROOT / 'data' / 'extract_audio_wav.py', '--dataset-mode', DATASET_MODE])
else:
    print('Skipping WAV extraction.')

if RUN_AUDIO_VGGISH:
    run_cmd([PYTHON, REPO_ROOT / 'data' / 'extract_vggish_embeddings.py', '--dataset-mode', DATASET_MODE])
else:
    print('Skipping VGGish extraction.')

## Stage 5: Optional multimodal cache build

This is not required for the final stable BART path, but it is useful if you want the older precomputed multimodal experiments.

In [ ]:
if RUN_MULTIMODAL_CACHE:
    run_cmd([PYTHON, REPO_ROOT / 'data' / 'extract_multimodal_embeddings.py', '--dataset-mode', DATASET_MODE])
else:
    print('Skipping multimodal cache build.')

## Stage 6: Train the recommended final model

In [ ]:
if RUN_TRAIN:
    run_cmd(train_command(DATASET_MODE))
else:
    print('Skipping training.')

## Stage 7: Category-level evaluation

In [ ]:
if RUN_CATEGORY_ANALYSIS:
    run_cmd(category_analysis_command(DATASET_MODE))
else:
    print('Skipping category analysis.')

## Stage 8: Optional ablation suite

This is separate from the main final-model pipeline.

In [ ]:
if RUN_ABLATIONS:
    run_cmd(ablation_command())
else:
    print('Skipping ablation suite.')

## Run everything in sequence

This cell respects the stage toggles from the configuration cell.

In [ ]:
if RUN_ALL_STEPS:
    if RUN_CAPTION_PREP:
        run_cmd([PYTHON, REPO_ROOT / 'data' / 'extract_captions.py', '--dataset-mode', DATASET_MODE])
        run_cmd([PYTHON, REPO_ROOT / 'data' / 'build_vocab.py', '--dataset-mode', DATASET_MODE])
    if RUN_TASCC:
        for cmd in tascc_commands(DATASET_MODE):
            run_cmd(cmd)
    if RUN_VISUAL_SPLIT:
        run_cmd(split_visual_command(DATASET_MODE))
    if RUN_AUDIO_WAV:
        run_cmd([PYTHON, REPO_ROOT / 'data' / 'extract_audio_wav.py', '--dataset-mode', DATASET_MODE])
    if RUN_AUDIO_VGGISH:
        run_cmd([PYTHON, REPO_ROOT / 'data' / 'extract_vggish_embeddings.py', '--dataset-mode', DATASET_MODE])
    if RUN_MULTIMODAL_CACHE:
        run_cmd([PYTHON, REPO_ROOT / 'data' / 'extract_multimodal_embeddings.py', '--dataset-mode', DATASET_MODE])
    if RUN_TRAIN:
        run_cmd(train_command(DATASET_MODE))
    if RUN_CATEGORY_ANALYSIS:
        run_cmd(category_analysis_command(DATASET_MODE))
    if RUN_ABLATIONS:
        run_cmd(ablation_command())
else:
    print('RUN_ALL_STEPS is False. Execute the individual stage cells you need.')

## Quick result inspection

In [ ]:
metrics_path = TRAIN_RUN_DIR / 'test_metrics_xe.json'
summary_path = TRAIN_RUN_DIR / 'summary.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(json.dumps(metrics, indent=2))
else:
    print(f'Metrics file not found yet: {metrics_path}')

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('\nSummary:')
    print(json.dumps(summary, indent=2))
